In [1]:
import requests
import pandas as pd
from tqdm import tqdm
import time


In [2]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

API_KEY = os.getenv("RAWG_API_KEY")

BASE_URL = "https://api.rawg.io/api/games"

In [3]:
def fetch_games(pages=80, page_size=40):
    all_games = []
    
    for page in tqdm(range(1, pages + 1)):
        params = {
            "key": API_KEY,
            "page": page,
            "page_size": page_size
        }
        
        response = requests.get(BASE_URL, params=params)
        
        if response.status_code != 200:
            print(f"Error en página {page}")
            continue
        
        data = response.json()
        all_games.extend(data["results"])
        
        time.sleep(1)
    
    return all_games

In [4]:
games = fetch_games(pages=80)
len(games)

 59%|█████▉    | 47/80 [02:52<02:11,  3.99s/it]

Error en página 46
Error en página 47


 60%|██████    | 48/80 [02:52<01:30,  2.84s/it]

Error en página 48


 62%|██████▎   | 50/80 [02:55<00:58,  1.96s/it]

Error en página 50


 66%|██████▋   | 53/80 [02:59<00:39,  1.46s/it]

Error en página 53


100%|██████████| 80/80 [04:48<00:00,  3.61s/it]


3000

In [ ]:
def build_raw_dataframe(games):
    rows = []
    
    for g in games:
        row = {}
        
        row["id"] = g.get("id")
        row["name"] = g.get("name")
        row["slug"] = g.get("slug")
        
        row["released"] = g.get("released")
        
        row["rating"] = g.get("rating")
        row["ratings_count"] = g.get("ratings_count")
        row["reviews_text_count"] = g.get("reviews_text_count")
        row["metacritic"] = g.get("metacritic")
        row["added"] = g.get("added")
        row["suggestions_count"] = g.get("suggestions_count")
        
        genres = [genre["name"] for genre in g.get("genres", [])]
        row["genres"] = ", ".join(genres)
        
        platforms = [p["platform"]["name"] for p in g.get("platforms", [])]
        row["platforms"] = ", ".join(platforms)
        
        devs = [d["name"] for d in g.get("developers", [])]
        row["developers"] = ", ".join(devs)
        
        pubs = [p["name"] for p in g.get("publishers", [])]
        row["publishers"] = ", ".join(pubs)
        
        rows.append(row)
    
    return pd.DataFrame(rows)

In [6]:
df_raw = build_raw_dataframe(games)
df_raw.head()

,id,name,slug,released,rating,ratings_count,reviews_text_count,metacritic,added,suggestions_count,genres,platforms,developers,publishers
0,3498,Grand Theft Auto V,grand-theft-auto-v,2013-09-17,4.47,7365,74,92.0,22548,450,Action,"PlayStation 5, Xbox Series S/X, PlayStation 3,...",,
1,3328,The Witcher 3: Wild Hunt,the-witcher-3-wild-hunt,2015-05-18,4.64,7180,83,92.0,22189,688,"Action, RPG","PlayStation 5, Xbox Series S/X, macOS, PlaySta...",,
2,4200,Portal 2,portal-2,2011-04-18,4.58,6090,40,95.0,20873,570,"Shooter, Puzzle","PlayStation 3, PC, Xbox 360, Linux, macOS, Xbo...",,
3,4291,Counter-Strike: Global Offensive,counter-strike-global-offensive,2012-08-21,3.57,3635,28,81.0,18357,607,Shooter,"PC, Linux, Xbox 360, PlayStation 3",,
4,5286,Tomb Raider (2013),tomb-raider,2013-03-05,4.06,4092,17,86.0,17809,664,Action,"PlayStation 3, Xbox 360, macOS, PC",,


In [7]:
df_raw = df_raw.drop_duplicates(subset="id")
df_raw = df_raw.reset_index(drop=True)

In [8]:
df_raw.to_csv("rawg_games_full_dataset.csv", index=False)